In [2]:
import os
import subprocess

mp3_dir = "outputs/mp3"
wav_dir = "outputs/wav"
os.makedirs(wav_dir, exist_ok=True)

base_name = input("请输入要转换的文件名（不含后缀）：").strip()

mp3_path = os.path.join(mp3_dir, base_name + ".mp3")
wav_path = os.path.join(wav_dir, base_name + ".wav")

if not os.path.exists(mp3_path):
    print("❌ 找不到该 MP3 文件")
else:
    cmd = [
        "ffmpeg",
        "-y",              # 覆盖输出
        "-loglevel", "error",  # ✅ 只显示错误，不刷屏
        "-i", mp3_path,
        "-ar", "16000",
        "-ac", "1",
        wav_path
    ]
    subprocess.run(cmd, check=True)
    print(f"✅ 转换完成：{wav_path}")

请输入要转换的文件名（不含后缀）：Mt_1_en
✅ 转换完成：outputs/wav/Mt_1_en.wav


In [9]:
import os
import subprocess

# 找到 aligner 环境路径
conda_prefix = subprocess.check_output(
    ["conda", "info", "--base"], text=True
).strip()

aligner_bin = os.path.join(conda_prefix, "envs", "aligner", "bin")
os.environ["PATH"] = aligner_bin + ":" + os.environ["PATH"]

# 验证
!mfa version

python(84858) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(84859) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


3.3.9


In [10]:
import os
import shutil

os.makedirs("corpus", exist_ok=True)

base = "Mt_1_en"

shutil.copy(f"outputs/wav/{base}.wav", f"corpus/{base}.wav")
shutil.copy(f"outputs/plaintext/{base}.txt", f"corpus/{base}.txt")

print("✅ corpus 准备完成")

✅ corpus 准备完成


In [11]:
!mfa align corpus \
  english_mfa \
  english_mfa \
  forcealign \
  --clean \
  --overwrite

python(84922) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
   1% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/100  [ 0:00:01 < -:--:-- , ? it/s ]
 INFO     Found 1 speaker across 1 file, average number of utterances per       
          speaker: 1.0                                                          
 INFO     Initializing multiprocessing jobs...                                  
 WARNING  Number of jobs was specified as 3, but due to only having 1 speakers, 
          MFA will only use 1 jobs. Use the --single_speaker flag if you would  
          like to split utterances across jobs regardless of their speaker.     
 INFO     Normalizing text...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Generating MFCCs...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━

In [18]:
# 连接数据库

import sqlite3

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

In [28]:
# 清空数据库时间戳

cursor.execute("""
UPDATE words
SET start_time = NULL,
    end_time = NULL;
""")
conn.commit()

In [34]:
import sqlite3
import re

def parse_textgrid(textgrid_path):
    intervals = []
    current = None
    in_words_tier = False

    with open(textgrid_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith('name = "words"'):
                in_words_tier = True
                continue
            if line.startswith('item ['):
                in_words_tier = False
                continue

            if not in_words_tier:
                continue

            # ✅ 区间开始
            if re.match(r'intervals\s*\[\d+\]:', line):
                current = {'xmin': None, 'xmax': None, 'text': ''}
                continue

            # ✅ 关键修复：current 不存在就跳过
            if current is None:
                continue

            if m := re.match(r'xmin\s*=\s*([\d\.]+)', line):
                current['xmin'] = float(m.group(1))
            elif m := re.match(r'xmax\s*=\s*([\d\.]+)', line):
                current['xmax'] = float(m.group(1))
            elif m := re.match(r'text\s*=\s*"([^"]*)"', line):
                word = m.group(1).strip()
                if word and current['xmin'] is not None:
                    intervals.append((
                        int(round(current['xmin'] * 1000)),
                        int(round(current['xmax'] * 1000)),
                        word
                    ))
                current = None  # ✅ 强制重置

    return intervals


def update_words_timestamps(db_path, textgrid_path):
    intervals = parse_textgrid(textgrid_path)
    if not intervals:
        print("❌ 未提取到单词时间戳")
        return

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        cursor.execute("SELECT id, word FROM words WHERE type = 'word' ORDER BY id")
        records = cursor.fetchall()

        idx = 0
        for word_id, word_text in records:
            if idx >= len(intervals):
                print("⚠️ TextGrid 单词不足")
                break

            start_ms, end_ms, tg_word = intervals[idx]

            if word_text.lower() != tg_word.lower():
                print(f"⚠️ 不匹配：DB={word_text} ≠ TG={tg_word}")
                idx += 1
                continue

            cursor.execute("""
                UPDATE words
                SET start_time = ?, end_time = ?
                WHERE id = ?
            """, (start_ms, end_ms, word_id))

            idx += 1

        conn.commit()
        print(f"✅ 成功更新 {idx} 个单词的时间戳（毫秒）")

    except Exception as e:
        print("❌ 更新失败：", e)
        conn.rollback()
    finally:
        conn.close()


# ------------------- 调用 -------------------
db_path = "db/bible.db"
textgrid_path = "forcealign/Mt_1_en.TextGrid"

update_words_timestamps(db_path, textgrid_path)

✅ 成功更新 533 个单词的时间戳（毫秒）


In [31]:
# drop列操作需要查看版本

import sqlite3
sqlite3.sqlite_version

'3.51.2'

In [32]:
# drop列

import sqlite3

conn = sqlite3.connect("db/bible.db")
cur = conn.cursor()

cur.execute("ALTER TABLE words DROP COLUMN start_time")
cur.execute("ALTER TABLE words DROP COLUMN end_time")
conn.commit()

In [33]:
# 添加时间戳列为整数类型

cur.execute("ALTER TABLE words ADD COLUMN start_time INTEGER")
cur.execute("ALTER TABLE words ADD COLUMN end_time INTEGER")
conn.commit()